In [3]:
import sys
import pandas as pd 
import numpy as np 

sys.path.append("src")

In [5]:
from analyse_exploratoire import display_missing_values, convert_data_types, impute_target_with_reg, is_outlier, missing_summary


In [6]:
df = pd.read_csv("data/data.csv")

In [7]:
df

,date,cheveux,age,exp,salaire,sexe,diplome,specialite,note,dispo,embauche
0,02/06/2012,roux,25.0,9.0,26803.0,F,licence,geologie,97.08,non,0
1,21/04/2011,blond,35.0,13.0,38166.0,M,licence,forage,63.86,non,0
2,07/09/2012,blond,29.0,13.0,35207.0,M,licence,geologie,78.50,non,0
3,01/07/2011,brun,NaN,12.0,32442.0,M,licence,geologie,45.09,non,0
4,07/08/2012,roux,35.0,6.0,28533.0,F,licence,detective,81.91,non,0
...,...,...,...,...,...,...,...,...,...,...,...
19995,10/03/2012,roux,47.0,9.0,35723.0,M,licence,geologie,66.47,non,0
19996,19/09/2010,chatain,38.0,10.0,33570.0,F,master,geologie,62.29,non,1
19997,02/09/2010,chatain,23.0,6.0,33751.0,F,doctorat,detective,103.48,oui,0
19998,06/12/2011,chatain,33.0,11.0,34167.0,F,licence,detective,73.35,non,0


In [12]:
df = convert_data_types(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        19909 non-null  object 
 1   cheveux     19895 non-null  object 
 2   age         19899 non-null  float64
 3   exp         19894 non-null  float64
 4   salaire     19895 non-null  float64
 5   sexe        19900 non-null  object 
 6   diplome     19891 non-null  object 
 7   specialite  19907 non-null  object 
 8   note        19886 non-null  float64
 9   dispo       19894 non-null  object 
 10  embauche    20000 non-null  string 
dtypes: float64(4), object(6), string(1)
memory usage: 1.7+ MB


In [ ]:
display_missing_values(df)


 Variables NUMÉRIQUES


age        0.505
exp        0.530
salaire    0.525
note       0.570
dtype: float64


 Variables CATÉGORIELLES 


date          0.455
cheveux       0.525
sexe          0.500
diplome       0.545
specialite    0.465
dispo         0.530
dtype: float64

,date,cheveux,age,exp,salaire,sexe,diplome,specialite,note,dispo,embauche
0,02/06/2012,roux,25.0,9.0,26803.0,F,licence,geologie,97.08,non,0
1,21/04/2011,blond,35.0,13.0,38166.0,M,licence,forage,63.86,non,0
2,07/09/2012,blond,29.0,13.0,35207.0,M,licence,geologie,78.50,non,0
3,01/07/2011,brun,NaN,12.0,32442.0,M,licence,geologie,45.09,non,0
4,07/08/2012,roux,35.0,6.0,28533.0,F,licence,detective,81.91,non,0
...,...,...,...,...,...,...,...,...,...,...,...
19995,10/03/2012,roux,47.0,9.0,35723.0,M,licence,geologie,66.47,non,0
19996,19/09/2010,chatain,38.0,10.0,33570.0,F,master,geologie,62.29,non,1
19997,02/09/2010,chatain,23.0,6.0,33751.0,F,doctorat,detective,103.48,oui,0
19998,06/12/2011,chatain,33.0,11.0,34167.0,F,licence,detective,73.35,non,0


In [17]:
missing_summary(df)

,Nb_valeurs_manquantes,Taux_valeurs_manquantes (%)
note,114,0.570
diplome,109,0.545
dispo,106,0.530
cheveux,105,0.525
sexe,100,0.500
exp,96,0.480
salaire,95,0.475
specialite,93,0.465
date,91,0.455
age,91,0.455


In [13]:
feature_map = {
    "salaire":   ["age", "exp", "note"],
    "age":       ["exp",  "salaire", "note"],
    "exp":       ["age", "salaire", "note"],
    "note":      ["salaire", "age", "exp"],
}

#  Application ciblée : uniquement si la colonne a des NaN
for target, feats in feature_map.items():
    if target in df.columns and df[target].isna().any():
        df = impute_target_with_reg(df, target, feats)

#  Contrôle
imputed_cols = [c for c in df.columns if c.endswith("_imp_reg")]
print("Colonnes imputées :", imputed_cols)
print("NaN restants sur colonnes imputées :")
print(df[imputed_cols].isna().sum())

Colonnes imputées : ['salaire_imp_reg', 'age_imp_reg', 'exp_imp_reg', 'note_imp_reg']
NaN restants sur colonnes imputées :
salaire_imp_reg    0
age_imp_reg        0
exp_imp_reg        0
note_imp_reg       0
dtype: int64


In [17]:
col_numericals = ["note", "exp", "age", "salaire"]
for col in col_numericals:
    df[col + "_outlier"] = is_outlier(df, col)

# Vérification rapide : nb d'outliers par variable
print(df[[c for c in df.columns if "_outlier" in c]].sum())

note_outlier       149
exp_outlier          8
age_outlier        215
salaire_outlier    123
dtype: int64
